# Partie 3 — Spark SQL et DataFrames · comparaison RDD / DataFrame

**Objectif** : refaire et enrichir l'analyse de la Partie 2 avec l'API structurée, puis comparer
les deux approches de façon **mesurée et reproductible**.

| Élément | Valeur |
|---|---|
| Événements | `s3a://lake/bronze/history/orders_history.jsonl` — **mêmes données que la Partie 2** (SHA-256 `57142d2d…`) |
| Référentiel | `data/restaurants.json` (200 restaurants, SHA-256 `8cac39b1…`) |
| Règles métier | identiques à la Partie 2 : `status = DELIVERED`, **dédoublonnage par `order_id`**, somme de `amount_mad` |
| Cluster | 2 workers, 4 executors × 2 cœurs, 8 partitions (vérifié par assertions) |

Convention de lecture : **📏 Mesuré** = résultat produit par ce notebook ; **📘 Théorie** = explication
générale (non mesurée ici).

> ⚠️ Données issues de la **reconstitution** de `producer.py` (fichier du professeur non fourni).

In [1]:
import json
import re
import socket
import statistics
import time
import urllib.request
from operator import add
from pathlib import Path

import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (DoubleType, IntegerType, StringType, StructField, StructType,
                               TimestampType)

from common.spark_session import get_spark, lake_path

spark = get_spark("part3-dataframe", cores_max=8)
sc = spark.sparkContext
sc.setLogLevel("WARN")

HISTORY = lake_path("bronze", "history", "orders_history.jsonl")
RESTAURANTS = "file:///workspace/data/restaurants.json"
SILVER = lake_path("silver", "orders_history")
LOGS = Path("/workspace/docs/logs")
PART2 = json.loads((LOGS / "09_part2_results.json").read_text(encoding="utf-8"))

API = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}"
rest = lambda path: json.load(urllib.request.urlopen(f"{API}/{path}"))

deadline = time.time() + 60
while sc._jsc.sc().getExecutorMemoryStatus().size() - 1 < 4 and time.time() < deadline:
    time.sleep(1)
executors_start = sorted(e["id"] for e in rest("executors") if e["id"] != "driver")

CONDITIONS = {
    "app_id": sc.applicationId, "spark_version": sc.version, "master": sc.master,
    "default_parallelism": sc.defaultParallelism, "executors": executors_start,
    "spark.sql.shuffle.partitions": spark.conf.get("spark.sql.shuffle.partitions"),
    "spark.sql.adaptive.enabled": spark.conf.get("spark.sql.adaptive.enabled"),
    "spark.sql.autoBroadcastJoinThreshold": spark.conf.get("spark.sql.autoBroadcastJoinThreshold"),
}
for k, v in CONDITIONS.items():
    print(f"{k:<40} {v}")
assert len(executors_start) == 4 and sc.defaultParallelism == 8
assert CONDITIONS["spark.sql.shuffle.partitions"] == "8"


def plan(df, mode="formatted"):
    # Équivalent texte de df.explain(mode) (pour l'enregistrer dans les logs)
    return spark._jvm.org.apache.spark.sql.api.python.PythonSQLUtils.explainString(df._jdf.queryExecution(), mode)


PLANS = {}

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/14 18:18:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


app_id                                   app-20261114181835-0005
spark_version                            3.5.9
master                                   spark://spark-master:7077
default_parallelism                      8
executors                                ['0', '1', '2', '3']
spark.sql.shuffle.partitions             8
spark.sql.adaptive.enabled               true
spark.sql.autoBroadcastJoinThreshold     10485760b


## Étape 1 — Lecture avec un schéma explicite (`StructType`), sans inférence

Le schéma est déclaré **avant** la lecture : Spark ne parcourt pas les données pour le deviner et chaque
colonne a d'emblée son type exact (`event_time` en `TimestampType`, `items_count` en `IntegerType`).
La vérification utilise le mode `PERMISSIVE` avec une colonne `_corrupt_record` : toute ligne non conforme
au schéma y serait capturée au lieu d'être silencieusement mise à `null`.

In [2]:
ORDER_SCHEMA = StructType([
    StructField("order_id", StringType(), nullable=False),
    StructField("restaurant_id", StringType(), nullable=False),
    StructField("customer_id", StringType(), nullable=False),
    StructField("city", StringType(), nullable=False),
    StructField("status", StringType(), nullable=False),
    StructField("amount_mad", DoubleType(), nullable=False),
    StructField("items_count", IntegerType(), nullable=False),
    StructField("distance_km", DoubleType(), nullable=False),
    StructField("courier_id", StringType(), nullable=False),
    StructField("event_time", TimestampType(), nullable=False),
])
RESTAURANT_SCHEMA = StructType([
    StructField("restaurant_id", StringType(), nullable=False),
    StructField("name", StringType(), nullable=False),
    StructField("city", StringType(), nullable=False),
    StructField("category", StringType(), nullable=False),
    StructField("latitude", DoubleType(), nullable=False),
    StructField("longitude", DoubleType(), nullable=False),
])

events = spark.read.schema(ORDER_SCHEMA).json(HISTORY)          # lazy : aucun job lancé
restaurants = spark.read.schema(RESTAURANT_SCHEMA).json(RESTAURANTS)
events.printSchema()
print("NB : les sources fichier (JSON, Parquet) exposent toujours nullable=true ; la non-nullité est donc vérifiée ci-dessous par comptage.")

# --- Vérification du schéma sur les données réelles ---------------------------
sc.setJobDescription("P3-1 vérification du schéma")
CHECK_SCHEMA = StructType(ORDER_SCHEMA.fields + [StructField("_corrupt_record", StringType())])  # copie : add() modifierait ORDER_SCHEMA
check = (spark.read.schema(CHECK_SCHEMA)
         .option("mode", "PERMISSIVE").json(HISTORY).cache())   # cache requis pour filtrer _corrupt_record
n_lines = check.count()
n_corrupt = check.filter(F.col("_corrupt_record").isNotNull()).count()
nulls = check.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in events.columns]).first().asDict()
bad_status = check.filter(~F.col("status").isin("CREATED", "PICKED_UP", "DELIVERED", "CANCELLED")).count()
check.unpersist()
n_rest = restaurants.count()

print(f"Lignes lues            : {n_lines}  (Partie 2 : 150281)")
print(f"Lignes non conformes   : {n_corrupt}")
print(f"Valeurs nulles         : {nulls}")
print(f"Statuts inconnus       : {bad_status}")
print(f"Restaurants            : {n_rest}")
print(f"Partitions de lecture  : {events.rdd.getNumPartitions()}  (RDD Partie 2 : 8)")
assert n_lines == 150281 and n_corrupt == 0 and sum(nulls.values()) == 0 and bad_status == 0 and n_rest == 200
assert events.rdd.getNumPartitions() == 8, "même nombre de partitions que le RDD exigé"

26/11/14 18:18:41 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- order_id: string (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount_mad: double (nullable = true)
 |-- items_count: integer (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)

NB : les sources fichier (JSON, Parquet) exposent toujours nullable=true ; la non-nullité est donc vérifiée ci-dessous par comptage.


Lignes lues            : 150281  (Partie 2 : 150281)
Lignes non conformes   : 0
Valeurs nulles         : {'order_id': 0, 'restaurant_id': 0, 'customer_id': 0, 'city': 0, 'status': 0, 'amount_mad': 0, 'items_count': 0, 'distance_km': 0, 'courier_id': 0, 'event_time': 0}
Statuts inconnus       : 0
Restaurants            : 200
Partitions de lecture  : 8  (RDD Partie 2 : 8)


### 📏 Coût de l'inférence de schéma (comparaison mesurée)

Sans schéma, `spark.read.json()` lance **immédiatement un job** qui lit tout le fichier pour deviner les
types (avant même la première action), et le résultat est moins précis.

In [3]:
def jobs_count():
    return len(rest("jobs"))


inference = {"explicite": [], "inference": []}
for i in range(3):
    n0 = jobs_count(); t0 = time.perf_counter()
    spark.read.schema(ORDER_SCHEMA).json(HISTORY)
    inference["explicite"].append((round(time.perf_counter() - t0, 3), jobs_count() - n0))
    sc.setJobDescription("P3-1b inférence de schéma")
    n0 = jobs_count(); t0 = time.perf_counter()
    inferred = spark.read.json(HISTORY)
    inference["inference"].append((round(time.perf_counter() - t0, 3), jobs_count() - n0))

for k, v in inference.items():
    print(f"{k:<10} : durées {[d for d, _ in v]} s | jobs lancés à la lecture : {[j for _, j in v]}")
print("\nSchéma inféré :")
inferred.printSchema()
explicit_types = dict((f.name, f.dataType.simpleString()) for f in ORDER_SCHEMA.fields)
inferred_types = dict((f.name, f.dataType.simpleString()) for f in inferred.schema.fields)
print("📏 Différences de type (explicite -> inféré) :",
      {c: f"{explicit_types[c]} -> {inferred_types[c]}" for c in explicit_types if explicit_types[c] != inferred_types.get(c)})
print("📏 Ordre des colonnes inféré :", inferred.columns)

explicite  : durées [0.033, 0.032, 0.058] s | jobs lancés à la lecture : [0, 0, 0]
inference  : durées [1.588, 0.532, 0.508] s | jobs lancés à la lecture : [1, 1, 1]

Schéma inféré :
root
 |-- amount_mad: double (nullable = true)
 |-- city: string (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- event_time: string (nullable = true)
 |-- items_count: long (nullable = true)
 |-- order_id: string (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- status: string (nullable = true)

📏 Différences de type (explicite -> inféré) : {'items_count': 'int -> bigint', 'event_time': 'timestamp -> string'}
📏 Ordre des colonnes inféré : ['amount_mad', 'city', 'courier_id', 'customer_id', 'distance_km', 'event_time', 'items_count', 'order_id', 'restaurant_id', 'status']


## Étape 2 — `filter` (DELIVERED) et dédoublonnage par `order_id`

`dropDuplicates(["order_id"])` est la traduction DataFrame du `reduceByKey(lambda a, b: a)` de la Partie 2.

In [4]:
delivered = events.filter(F.col("status") == "DELIVERED")
delivered_unique = delivered.dropDuplicates(["order_id"])

sc.setJobDescription("P3-2 filtre DELIVERED + dédoublonnage")
n_delivered_raw = delivered.count()
n_delivered = delivered_unique.count()
print(f"DELIVERED (brut)      : {n_delivered_raw}   (Partie 2 : 46977)")
print(f"DELIVERED (uniques)   : {n_delivered}   (Partie 2 : 46028)")
print(f"Doublons écartés      : {n_delivered_raw - n_delivered}   (Partie 2 : 949)")
assert (n_delivered_raw, n_delivered) == (46977, 46028)

DELIVERED (brut)      : 46977   (Partie 2 : 46977)
DELIVERED (uniques)   : 46028   (Partie 2 : 46028)
Doublons écartés      : 949   (Partie 2 : 949)


## Étape 3 — CA et panier moyen par ville : API DataFrame puis SQL

Une seule agrégation (`groupBy` + `agg`) calcule CA, nombre de commandes et panier moyen
(équivalent de l'`aggregateByKey` « un seul passage » de la Partie 2).

In [5]:
ca_city_df = (delivered_unique
              .groupBy("city")
              .agg(F.sum("amount_mad").alias("ca_mad"),
                   F.count(F.lit(1)).alias("commandes"),
                   F.avg("amount_mad").alias("panier_moyen_mad"))
              .orderBy(F.desc("ca_mad")))

sc.setJobDescription("P3-3a CA + panier moyen par ville (DataFrame)")
ca_city = ca_city_df.collect()
total = sum(r.ca_mad for r in ca_city)
ca_city_df.select("city", F.format_number("ca_mad", 2).alias("ca_mad"),
                  (F.col("ca_mad") / total * 100).cast("decimal(5,1)").alias("part_%"), "commandes",
                  F.round("panier_moyen_mad", 2).alias("panier_moyen_mad")).show(truncate=False)
print(f"CA total : {total:,.2f} MAD (Partie 2 : {PART2['ca_total_mad']:,.2f})")

+----------+------------+------+---------+----------------+
|city      |ca_mad      |part_%|commandes|panier_moyen_mad|
+----------+------------+------+---------+----------------+
|Casablanca|2,441,036.72|27.7  |12592    |193.86          |
|Marrakech |1,252,086.80|14.2  |6399     |195.67          |
|Agadir    |1,066,986.41|12.1  |5407     |197.33          |
|Rabat     |1,003,432.78|11.4  |5498     |182.51          |
|Tanger    |953,252.19  |10.8  |4893     |194.82          |
|Fes       |872,019.55  |9.9   |4599     |189.61          |
|Oujda     |833,093.75  |9.5   |4594     |181.34          |
|Meknes    |377,737.33  |4.3   |2046     |184.62          |
+----------+------------+------+---------+----------------+

CA total : 8,799,645.53 MAD (Partie 2 : 8,799,645.53)


In [6]:
events.createOrReplaceTempView("orders_history")
SQL_CA = '''
WITH delivered AS (                       -- dédoublonnage par order_id
    SELECT order_id, first(city) AS city, first(amount_mad) AS amount_mad
    FROM orders_history
    WHERE status = 'DELIVERED'
    GROUP BY order_id
)
SELECT city,
       sum(amount_mad)  AS ca_mad,
       count(*)         AS commandes,
       avg(amount_mad)  AS panier_moyen_mad
FROM delivered
GROUP BY city
ORDER BY ca_mad DESC
'''
sc.setJobDescription("P3-3b CA + panier moyen par ville (SQL)")
ca_city_sql = spark.sql(SQL_CA).collect()

# --- Contrôles : DataFrame = SQL = Partie 2 (RDD) ----------------------------------
df_d = {r.city: r for r in ca_city}
sql_d = {r.city: r for r in ca_city_sql}
same_sql = all(abs(df_d[c].ca_mad - sql_d[c].ca_mad) < 1e-6 and df_d[c].commandes == sql_d[c].commandes for c in df_d)
same_p2_ca = all(abs(df_d[c].ca_mad - v) < 0.01 for c, v in PART2["ca_par_ville_mad"].items())
same_p2_avg = all(abs(df_d[c].panier_moyen_mad - v["panier_moyen_mad"]) < 0.01 and df_d[c].commandes == v["commandes"]
                  for c, v in PART2["panier_moyen_par_ville"].items())
print(f"DataFrame == SQL                      : {same_sql}")
print(f"CA identique à la Partie 2 (RDD)      : {same_p2_ca}")
print(f"Panier moyen identique à la Partie 2  : {same_p2_avg}")
assert same_sql and same_p2_ca and same_p2_avg

DataFrame == SQL                      : True
CA identique à la Partie 2 (RDD)      : True
Panier moyen identique à la Partie 2  : True


## Étape 4 — Jointure avec le référentiel : CA par catégorie et par ville

Le référentiel (200 lignes, quelques Ko) est bien plus petit que `spark.sql.autoBroadcastJoinThreshold`
(10 Mo) : Catalyst doit donc choisir un **BroadcastHashJoin** sans indication de notre part.

In [7]:
enriched = delivered_unique.join(
    restaurants.select("restaurant_id", F.col("name").alias("restaurant_name"), "category"),
    on="restaurant_id", how="inner")

ca_cat_city_df = (enriched.groupBy("category", "city")
                  .agg(F.round(F.sum("amount_mad"), 2).alias("ca_mad"),
                       F.count(F.lit(1)).alias("commandes"))
                  .orderBy(F.desc("ca_mad")))

sc.setJobDescription("P3-4 jointure + CA par catégorie et ville")
ca_cat_city = ca_cat_city_df.collect()
assert sum(r.commandes for r in ca_cat_city) == n_delivered, "la jointure a perdu ou dupliqué des commandes"
print(f"{len(ca_cat_city)} couples (catégorie, ville) — {n_delivered} commandes conservées par la jointure\n")
ca_cat_city_df.show(10, truncate=False)

print("CA par catégorie (toutes villes) :")
(enriched.groupBy("category").agg(F.round(F.sum("amount_mad"), 2).alias("ca_mad"),
                                  F.count(F.lit(1)).alias("commandes"),
                                  F.round(F.avg("amount_mad"), 2).alias("panier_moyen_mad"))
 .orderBy(F.desc("ca_mad")).show(truncate=False))

print("Matrice CA (k MAD) catégorie × ville :")
(enriched.groupBy("category").pivot("city").agg(F.round(F.sum("amount_mad") / 1000, 1))
 .orderBy("category").show(truncate=False))

57 couples (catégorie, ville) — 46028 commandes conservées par la jointure



+--------+----------+---------+---------+
|category|city      |ca_mad   |commandes|
+--------+----------+---------+---------+
|marocain|Casablanca|726167.38|3320     |
|marocain|Agadir    |570645.26|2561     |
|pizza   |Marrakech |402143.88|1891     |
|pizza   |Rabat     |363823.82|1721     |
|sushi   |Casablanca|363140.33|1286     |
|marocain|Marrakech |362085.77|1691     |
|pizza   |Casablanca|353738.15|1678     |
|pizza   |Fes       |346413.76|1629     |
|burger  |Casablanca|340428.01|1955     |
|pizza   |Tanger    |335594.38|1601     |
+--------+----------+---------+---------+
only showing top 10 rows

CA par catégorie (toutes villes) :


+----------+----------+---------+----------------+
|category  |ca_mad    |commandes|panier_moyen_mad|
+----------+----------+---------+----------------+
|pizza     |2361305.52|11208    |210.68          |
|marocain  |2330466.93|10677    |218.27          |
|sushi     |1171872.08|4176     |280.62          |
|burger    |875595.82 |5060     |173.04          |
|healthy   |623056.25 |3123     |199.51          |
|tacos     |604436.22 |4282     |141.16          |
|shawarma  |598537.89 |4991     |119.92          |
|patisserie|234374.82 |2511     |93.34           |
+----------+----------+---------+----------------+

Matrice CA (k MAD) catégorie × ville :


+----------+------+----------+-----+---------+------+-----+-----+------+
|category  |Agadir|Casablanca|Fes  |Marrakech|Meknes|Oujda|Rabat|Tanger|
+----------+------+----------+-----+---------+------+-----+-----+------+
|burger    |NULL  |340.4     |NULL |89.2     |96.2  |NULL |127.2|222.5 |
|healthy   |32.8  |308.1     |50.9 |13.4     |NULL  |67.9 |122.1|27.8  |
|marocain  |570.6 |726.2     |189.5|362.1    |82.4  |278.5|79.1 |42.1  |
|patisserie|43.8  |15.0      |24.8 |NULL     |NULL  |59.4 |63.4 |27.9  |
|pizza     |179.2 |353.7     |346.4|402.1    |137.6 |242.8|363.8|335.6 |
|shawarma  |76.3  |196.7     |38.0 |52.5     |25.2  |102.6|80.0 |27.2  |
|sushi     |119.7 |363.1     |98.4 |157.0    |NULL  |68.6 |141.4|223.6 |
|tacos     |44.6  |137.7     |124.0|175.8    |36.2  |13.1 |26.3 |46.6  |
+----------+------+----------+-----+---------+------+-----+-----+------+



### `explain(True)` sur la jointure : plans logique, optimisé et physique

La requête a été exécutée ci-dessus : avec AQE, le plan physique affiché est le **plan final**
(`isFinalPlan=true`), c'est-à-dire celui réellement exécuté.

In [8]:
ca_cat_city_df.explain(True)
PLANS["jointure_explain_true"] = plan(ca_cat_city_df, "extended")
PLANS["jointure_formatted"] = plan(ca_cat_city_df, "formatted")
phys = PLANS["jointure_explain_true"].split("== Physical Plan ==")[1]
print("\n📏 Stratégie de jointure physique :",
      "BroadcastHashJoin" if "BroadcastHashJoin" in phys else "SortMergeJoin" if "SortMergeJoin" in phys else "autre")

== Parsed Logical Plan ==
'Sort ['ca_mad DESC NULLS LAST], true
+- Aggregate [category#23, city#3], [category#23, city#3, round(sum(amount_mad#5), 2) AS ca_mad#1840, count(1) AS commandes#1842L]
   +- Project [restaurant_id#1, order_id#0, customer_id#2, city#3, status#4, amount_mad#5, items_count#6, distance_km#7, courier_id#8, event_time#9, restaurant_name#1810, category#23]
      +- Join Inner, (restaurant_id#1 = restaurant_id#20)
         :- Deduplicate [order_id#0]
         :  +- Filter (status#4 = DELIVERED)
         :     +- Relation [order_id#0,restaurant_id#1,customer_id#2,city#3,status#4,amount_mad#5,items_count#6,distance_km#7,courier_id#8,event_time#9] json
         +- Project [restaurant_id#20, name#21 AS restaurant_name#1810, category#23]
            +- Relation [restaurant_id#20,name#21,city#22,category#23,latitude#24,longitude#25] json

== Analyzed Logical Plan ==
category: string, city: string, ca_mad: double, commandes: bigint
Sort [ca_mad#1840 DESC NULLS LAST], true
+

### 📏 Comparaison BroadcastHashJoin vs SortMergeJoin (hint `merge`)

On force la stratégie alternative pour mesurer ce que le choix de Catalyst apporte.

In [9]:
def join_query(strategy):
    rest_side = restaurants.select("restaurant_id", "category")
    rest_side = rest_side.hint(strategy) if strategy else rest_side
    return (delivered_unique.join(rest_side, "restaurant_id")
            .groupBy("category", "city").agg(F.sum("amount_mad").alias("ca_mad")))


join_times = {"BroadcastHashJoin (choix Catalyst)": [], "SortMergeJoin (hint merge)": []}
strategies = {"BroadcastHashJoin (choix Catalyst)": None, "SortMergeJoin (hint merge)": "merge"}
for label, s in strategies.items():                       # chauffe
    join_query(s).collect()
for i in range(5):
    order = list(strategies) if i % 2 == 0 else list(strategies)[::-1]
    for label in order:
        sc.setJobDescription(f"P3-4b jointure {label}")
        t0 = time.perf_counter(); join_query(strategies[label]).collect()
        join_times[label].append(round(time.perf_counter() - t0, 3))
smj = join_query("merge")
smj.collect()
PLANS["jointure_sortmerge_formatted"] = plan(smj, "formatted")
assert "SortMergeJoin" in PLANS["jointure_sortmerge_formatted"]
for k, v in join_times.items():
    print(f"{k:<36} médiane {statistics.median(v):.3f} s   {v}")

BroadcastHashJoin (choix Catalyst)   médiane 0.450 s   [0.776, 0.45, 0.392, 0.493, 0.414]
SortMergeJoin (hint merge)           médiane 0.492 s   [0.587, 0.492, 0.436, 0.493, 0.444]


## Étape 5 — Top 5 des restaurants (jointure + `orderBy` + `limit`)

In [10]:
top5_df = (delivered_unique.groupBy("restaurant_id")
           .agg(F.count(F.lit(1)).alias("commandes_livrees"), F.round(F.sum("amount_mad"), 2).alias("ca_mad"))
           .join(restaurants.select("restaurant_id", "name", "city", "category"), "restaurant_id")
           .orderBy(F.desc("commandes_livrees"), F.asc("restaurant_id"))
           .limit(5))
sc.setJobDescription("P3-5 top 5 restaurants")
top5 = top5_df.collect()
top5_df.show(truncate=False)

p2_top = PART2["top5_restaurants"]
same_top = (len(top5) == 5 and all(r.restaurant_id == t["restaurant_id"] and r.commandes_livrees == t["commandes_livrees"]
                                   and abs(r.ca_mad - t["ca_mad"]) < 0.011 for r, t in zip(top5, p2_top)))
print("Top 5 identique à la Partie 2 (RDD) :", same_top)
assert same_top

+-------------+-----------------+---------+-----------------------+----------+--------+
|restaurant_id|commandes_livrees|ca_mad   |name                   |city      |category|
+-------------+-----------------+---------+-----------------------+----------+--------+
|R-157        |1863             |416203.57|Bab Fassia Dakhla      |Agadir    |marocain|
|R-188        |1157             |242785.2 |Forno Roma Lazaret     |Oujda     |pizza   |
|R-013        |904              |106559.56|Shawarma Express Maarif|Casablanca|shawarma|
|R-147        |890              |154244.92|Smash Spot Iberia      |Tanger    |burger  |
|R-012        |728              |152653.9 |Forno Mamma Oasis      |Casablanca|pizza   |
+-------------+-----------------+---------+-----------------------+----------+--------+

Top 5 identique à la Partie 2 (RDD) : True


## Étape 6 — UDF `distance_band` vs `when/otherwise`

Règle : **court** (< 2 km), **moyen** (2 à 5 km), **long** (> 5 km). Trois implémentations :
1. **UDF Python** (`@udf`) : chaque ligne est sérialisée vers un processus Python puis renvoyée à la JVM ;
2. **UDF pandas** (`@pandas_udf`) : même idée mais par lots vectorisés via Apache Arrow ;
3. **`when/otherwise`** : expression native Catalyst, compilée en bytecode JVM (Tungsten), sans Python.

Benchmark sur les 150 281 événements **déjà en cache** pour isoler le coût de la fonction elle-même.

In [11]:
@F.udf(returnType=StringType())
def distance_band_udf(d):
    if d is None:
        return None
    return "court" if d < 2 else "moyen" if d <= 5 else "long"


@F.pandas_udf(StringType())
def distance_band_pandas(d: pd.Series) -> pd.Series:
    out = pd.Series(np.select([d < 2, d <= 5], ["court", "moyen"], default="long"), index=d.index)
    return out.where(d.notna(), None)


def distance_band_native(col):
    return (F.when(col < 2, "court")
             .when(col <= 5, "moyen")
             .when(col > 5, "long"))                     # null si distance nulle


events_cached = events.cache()
sc.setJobDescription("P3-6 mise en cache des événements")
events_cached.count()

variants = {
    "UDF Python": lambda: distance_band_udf(F.col("distance_km")),
    "UDF pandas (Arrow)": lambda: distance_band_pandas(F.col("distance_km")),
    "when/otherwise (natif)": lambda: distance_band_native(F.col("distance_km")),
}
band_query = lambda make: events_cached.withColumn("distance_band", make()).groupBy("distance_band").count()

results_band = {k: dict(band_query(v).collect()) for k, v in variants.items()}   # chauffe + contrôle
assert results_band["UDF Python"] == results_band["UDF pandas (Arrow)"] == results_band["when/otherwise (natif)"]
print("Résultat identique pour les 3 variantes :", results_band["when/otherwise (natif)"])

udf_times = {k: [] for k in variants}
for i in range(5):
    names = list(variants)
    names = names[i % 3:] + names[:i % 3]                     # rotation de l'ordre
    for k in names:
        sc.setJobDescription(f"P3-6 distance_band {k}")
        t0 = time.perf_counter(); band_query(variants[k]).collect()
        udf_times[k].append(round(time.perf_counter() - t0, 3))

for k, v in udf_times.items():
    print(f"{k:<24} médiane {statistics.median(v):.3f} s   moyenne {statistics.mean(v):.3f} s   {v}")

for k, v in variants.items():
    PLANS[f"distance_band_{k}"] = plan(band_query(v), "formatted")
markers = {k: [m for m in ("BatchEvalPython", "ArrowEvalPython") if m in PLANS[f"distance_band_{k}"]] for k in variants}
print("\n📏 Opérateurs Python dans les plans physiques :", markers)

Résultat identique pour les 3 variantes : {'long': 38467, 'court': 40558, 'moyen': 71256}


UDF Python               médiane 0.346 s   moyenne 0.359 s   [0.426, 0.333, 0.319, 0.346, 0.37]
UDF pandas (Arrow)       médiane 0.281 s   moyenne 0.308 s   [0.269, 0.271, 0.281, 0.382, 0.338]
when/otherwise (natif)   médiane 0.134 s   moyenne 0.147 s   [0.131, 0.134, 0.133, 0.165, 0.171]

📏 Opérateurs Python dans les plans physiques : {'UDF Python': ['BatchEvalPython'], 'UDF pandas (Arrow)': ['ArrowEvalPython'], 'when/otherwise (natif)': []}


## Étape 7 — Couche Silver : Parquet partitionné par `city`

Résultat **nettoyé et enrichi** de l'historique :
- doublons exacts supprimés (`order_id`, `status`) — les doublons du générateur ;
- `event_time` typé (`timestamp`), ajout de `event_date` ;
- enrichissement par le référentiel (`restaurant_name`, `category`) et `distance_band` (version native).

Écrit dans `s3a://lake/silver/orders_history/`, partitionné par `city`.

In [12]:
silver_df = (events
             .dropDuplicates(["order_id", "status"])
             .join(F.broadcast(restaurants.select("restaurant_id", F.col("name").alias("restaurant_name"), "category")),
                   "restaurant_id")
             .withColumn("event_date", F.to_date("event_time"))
             .withColumn("distance_band", distance_band_native(F.col("distance_km"))))

sc.setJobDescription("P3-7 écriture Silver Parquet partitionné par city")
t0 = time.perf_counter()
silver_df.write.mode("overwrite").partitionBy("city").parquet(SILVER)
silver_write_s = round(time.perf_counter() - t0, 3)

sc.setJobDescription("P3-7 relecture Silver")
silver_back = spark.read.parquet(SILVER)
n_silver = silver_back.count()
print(f"Écriture : {silver_write_s} s — lignes relues : {n_silver} (attendu 150281 - 3037 doublons = 147244)")
assert n_silver == 147244
silver_back.groupBy("city").count().orderBy("city").show()
silver_back.printSchema()

# Élagage de partitions : une requête sur une ville ne lit que son dossier
casa = silver_back.filter(F.col("city") == "Casablanca")
PLANS["silver_partition_pruning"] = plan(casa, "formatted")
print([l.strip() for l in PLANS["silver_partition_pruning"].splitlines() if "PartitionFilters" in l])

Écriture : 9.342 s — lignes relues : 147244 (attendu 150281 - 3037 doublons = 147244)


+----------+-----+
|      city|count|
+----------+-----+
|    Agadir|17299|
|Casablanca|40304|
|       Fes|14734|
| Marrakech|20489|
|    Meknes| 6534|
|     Oujda|14666|
|     Rabat|17575|
|    Tanger|15643|
+----------+-----+

root
 |-- restaurant_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount_mad: double (nullable = true)
 |-- items_count: integer (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- restaurant_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- event_date: date (nullable = true)
 |-- distance_band: string (nullable = true)
 |-- city: string (nullable = true)

['PartitionFilters: [isnotnull(city#10153), (city#10153 = Casablanca)]']


## Étape 8 — Benchmark comparatif RDD / DataFrame / SQL

**Protocole** (identique pour toutes les variantes) :
- même SparkSession, mêmes 4 executors (vérifié avant/après), 8 partitions en lecture ;
- mêmes données, même logique : `DELIVERED` → dédoublonnage `order_id` → somme `amount_mad` par ville ;
- résultat de **chaque** exécution comparé à la Partie 2 ;
- 1 exécution de chauffe par variante, puis **5 passes entrelacées** avec rotation de l'ordre
  (neutralise la dérive : caches OS, JIT, connexions S3) ;
- tâches, stages, temps CPU executor et octets de shuffle relevés via l'API REST de Spark.

**Scénario A — de bout en bout depuis S3, sans cache** (même mesure que le benchmark de la Partie 2) :

| Variante | Détail |
|---|---|
| RDD | code de la Partie 2 : `textFile` → `json.loads` (Python) → `filter` → `reduceByKey` ×2 |
| DataFrame | `read.schema().json()` → `filter` → `dropDuplicates` → `groupBy().agg()` (AQE activé, défaut) |
| DataFrame sans AQE | idem, AQE désactivé : **8 partitions de shuffle exactement** comme le RDD |
| SQL | même calcul en `spark.sql` |

**Scénario B — données déjà en mémoire** : RDD d'événements parsés en cache vs DataFrame en cache.
Il isole le coût du **calcul** (filtre + dédoublonnage + agrégation) de celui de la **lecture/parsing JSON**.

In [13]:
def parse(line):
    try:
        return [json.loads(line)]
    except ValueError:
        return []


def rdd_ca():                                    # identique au benchmark de la Partie 2
    return dict(sc.textFile(HISTORY, minPartitions=8)
                .flatMap(parse)
                .filter(lambda e: e["status"] == "DELIVERED")
                .map(lambda e: (e["order_id"], (e["city"], e["amount_mad"])))
                .reduceByKey(lambda a, b: a)
                .values()
                .reduceByKey(add)
                .collect())


def df_query():
    return (spark.read.schema(ORDER_SCHEMA).json(HISTORY)
            .filter(F.col("status") == "DELIVERED")
            .dropDuplicates(["order_id"])
            .groupBy("city").agg(F.sum("amount_mad").alias("ca_mad")))


def df_ca():
    return {r.city: r.ca_mad for r in df_query().collect()}


def df_ca_no_aqe():
    spark.conf.set("spark.sql.adaptive.enabled", "false")
    try:
        return df_ca()
    finally:
        spark.conf.set("spark.sql.adaptive.enabled", "true")


def sql_ca():
    spark.read.schema(ORDER_SCHEMA).json(HISTORY).createOrReplaceTempView("orders_bench")
    return {r.city: r.ca_mad for r in spark.sql(
        "SELECT city, sum(amount_mad) AS ca_mad FROM (SELECT order_id, first(city) AS city, "
        "first(amount_mad) AS amount_mad FROM orders_bench WHERE status = 'DELIVERED' "
        "GROUP BY order_id) GROUP BY city").collect()}


# Scénario B : sources déjà en mémoire
rdd_events_cached = sc.textFile(HISTORY, minPartitions=8).flatMap(parse)   # mis en cache juste avant le scénario B


def rdd_ca_cached():
    return dict(rdd_events_cached
                .filter(lambda e: e["status"] == "DELIVERED")
                .map(lambda e: (e["order_id"], (e["city"], e["amount_mad"])))
                .reduceByKey(lambda a, b: a).values().reduceByKey(add).collect())


def df_ca_cached():
    return {r.city: r.ca_mad for r in events_cached.filter(F.col("status") == "DELIVERED")
            .dropDuplicates(["order_id"]).groupBy("city").agg(F.sum("amount_mad").alias("ca_mad")).collect()}


EXPECTED = PART2["ca_par_ville_mad"]


def check_result(res):
    assert set(res) == set(EXPECTED) and all(abs(res[c] - EXPECTED[c]) < 0.01 for c in EXPECTED), res


def run_benchmark(scenario, variants, rounds=5):
    times = {k: [] for k in variants}
    for k, fn in variants.items():                          # chauffe (non mesurée)
        sc.setJobGroup(f"warmup-{scenario}-{k}", f"P3-8 chauffe {scenario} {k}")
        check_result(fn())
    for i in range(rounds):
        names = list(variants)
        names = names[i % len(names):] + names[:i % len(names)]
        for k in names:
            sc.setJobGroup(f"bench-{scenario}-{k}", f"P3-8 bench {scenario} {k} (passe {i + 1})")
            t0 = time.perf_counter()
            res = variants[k]()
            times[k].append(round(time.perf_counter() - t0, 3))
            check_result(res)
    sc._jsc.clearJobGroup()
    return times


# --- Scénario A : AUCUNE donnée en cache ------------------------------------------
# Piège évité : le CacheManager de Spark substitue automatiquement un DataFrame en cache
# dès qu'une requête a le MÊME plan logique (même fichier, même schéma). On vide donc
# tous les caches et on vérifie dans le plan physique que la lecture se fait bien sur S3.
events_cached.unpersist(blocking=True)
rdd_events_cached.unpersist(blocking=True)
spark.catalog.clearCache()
for name, q in [("DataFrame", df_query()),
                ("SQL", spark.read.schema(ORDER_SCHEMA).json(HISTORY))]:
    p_ = plan(q, "formatted")
    assert "InMemoryTableScan" not in p_ and "Scan json" in p_, f"{name} lirait le cache"
print("✔ Scénario A : plans vérifiés, lecture S3 (aucun InMemoryTableScan)")
bench_A = run_benchmark("A", {"RDD": rdd_ca, "DataFrame": df_ca, "DataFrame sans AQE": df_ca_no_aqe, "SQL": sql_ca})

# --- Scénario B : sources en mémoire --------------------------------------------
rdd_events_cached = sc.textFile(HISTORY, minPartitions=8).flatMap(parse).cache()
events_cached = events.cache()
sc.setJobDescription("P3-8 mise en cache (scénario B)")
assert rdd_events_cached.count() == 150281 and events_cached.count() == 150281
p_ = plan(events_cached.filter(F.col("status") == "DELIVERED"), "formatted")
assert "InMemoryTableScan" in p_
print("✔ Scénario B : plans vérifiés, lecture depuis le cache (InMemoryTableScan)")
bench_B = run_benchmark("B", {"RDD (cache)": rdd_ca_cached, "DataFrame (cache)": df_ca_cached})

# Caches vidés : l'étape 9 doit montrer le plan de lecture S3 réel
events_cached.unpersist(blocking=True)
rdd_events_cached.unpersist(blocking=True)
spark.catalog.clearCache()
print("Toutes les exécutions (chauffe + 5 passes) ont produit le CA de la Partie 2 ✔")

✔ Scénario A : plans vérifiés, lecture S3 (aucun InMemoryTableScan)


✔ Scénario B : plans vérifiés, lecture depuis le cache (InMemoryTableScan)


Toutes les exécutions (chauffe + 5 passes) ont produit le CA de la Partie 2 ✔


In [14]:
def group_metrics(group, runs=5):
    # Agrège jobs / stages / tâches / CPU / shuffle des jobs d'un groupe, ramenés à UNE exécution
    jobs = [j for j in rest("jobs") if j.get("jobGroup") == group]
    stages = {}
    for s in rest("stages"):
        stages.setdefault(s["stageId"], []).append(s)
    done = skipped = tasks = cpu_ms = in_b = sh_r = sh_w = 0
    for j in jobs:
        for sid in j["stageIds"]:
            for att in stages.get(sid, []):
                if att["status"] == "COMPLETE":
                    done += 1; tasks += att["numCompleteTasks"]; cpu_ms += att["executorRunTime"]
                    in_b += att["inputBytes"]; sh_r += att["shuffleReadBytes"]; sh_w += att["shuffleWriteBytes"]
                elif att["status"] == "SKIPPED":
                    skipped += 1
    return {"jobs": len(jobs) / runs, "stages": done / runs, "stages_ignores": skipped / runs,
            "taches": tasks / runs, "temps_executors_s": round(cpu_ms / runs / 1000, 3),
            "lecture_Mo": round(in_b / runs / 1e6, 2), "shuffle_lu_Ko": round(sh_r / runs / 1024, 1),
            "shuffle_ecrit_Ko": round(sh_w / runs / 1024, 1)}


def summarize(scenario, times, ref):
    out = {}
    for k, v in times.items():
        out[k] = {"runs_s": v, "mediane_s": round(statistics.median(v), 3), "moyenne_s": round(statistics.mean(v), 3),
                  "ecart_type_s": round(statistics.stdev(v), 3), "min_s": min(v), "max_s": max(v),
                  **group_metrics(f"bench-{scenario}-{k}")}
    r = out[ref]["mediane_s"]
    for k in out:
        out[k]["acceleration_vs_" + ref.split()[0]] = round(r / out[k]["mediane_s"], 2)
    return out


BENCH = {"A_bout_en_bout_S3": summarize("A", bench_A, "RDD"),
         "B_donnees_en_cache": summarize("B", bench_B, "RDD (cache)")}

cols = ["mediane_s", "moyenne_s", "ecart_type_s", "jobs", "stages", "stages_ignores", "taches",
        "temps_executors_s", "lecture_Mo", "shuffle_ecrit_Ko"]
for scen, res in BENCH.items():
    print(f"\n📏 Scénario {scen}")
    print(f"{'Variante':<20}" + "".join(f"{c:>17}" for c in cols) + f"{'accélération':>14}")
    for k, m in res.items():
        acc = [v for kk, v in m.items() if kk.startswith("acceleration")][0]
        print(f"{k:<20}" + "".join(f"{m[c]:>17}" for c in cols) + f"{acc:>13}x")

executors_end = sorted(e["id"] for e in rest("executors") if e["id"] != "driver")
assert executors_end == executors_start, "les executors ont changé pendant le benchmark"
print("\nExecutors identiques avant/après le benchmark :", executors_end)


📏 Scénario A_bout_en_bout_S3
Variante                    mediane_s        moyenne_s     ecart_type_s             jobs           stages   stages_ignores           tachestemps_executors_s       lecture_Mo shuffle_ecrit_Ko  accélération
RDD                             0.847            0.848            0.151              1.0              3.0              0.0             24.0            5.043            34.42            579.5          1.0x
DataFrame                       0.369            0.347            0.041              3.0              3.0              3.0             10.0             0.93            34.42            837.9          2.3x
DataFrame sans AQE              0.335            0.337            0.046              1.0              3.0              0.0             24.0             1.18            34.42            841.1         2.53x
SQL                             0.357            0.373            0.046              3.0              3.0              3.0             10.0           

## Étape 9 — Lineage RDD vs plan d'exécution DataFrame

Le **lineage RDD** décrit *comment* calculer (une chaîne de fonctions Python opaques pour Spark).
Le **plan DataFrame** décrit *quoi* calculer : Catalyst peut le réécrire. Points à repérer dans le plan :
- `PushedFilters` : le filtre `status = 'DELIVERED'` est poussé dans le lecteur JSON ;
- `ReadSchema` : seules les colonnes utiles sont matérialisées (élagage de colonnes) ;
- `HashAggregate (partial)` puis `Exchange` puis `HashAggregate (final)` : pré-agrégation avant le shuffle ;
- les opérateurs préfixés `*(n)` / `WholeStageCodegen (n)` : code JVM généré par Tungsten.

In [15]:
lin = (sc.textFile(HISTORY, minPartitions=8).flatMap(parse)
       .filter(lambda e: e["status"] == "DELIVERED")
       .map(lambda e: (e["order_id"], (e["city"], e["amount_mad"])))
       .reduceByKey(lambda a, b: a).values().reduceByKey(add)).toDebugString()
PLANS["rdd_lineage"] = lin.decode() if isinstance(lin, bytes) else lin
print("===== Lineage RDD (Partie 2)\n" + PLANS["rdd_lineage"])

q = df_query()
sc.setJobDescription("P3-9 exécution pour plan final")
q.collect()
print("\n===== Plan DataFrame — explain('formatted')")
q.explain("formatted")
PLANS["dataframe_formatted"] = plan(q, "formatted")
PLANS["dataframe_extended"] = plan(q, "extended")

assert "InMemoryTableScan" not in PLANS["dataframe_formatted"], "le plan de l'étape 9 doit lire S3"
pf = [l.strip() for l in PLANS["dataframe_formatted"].splitlines() if l.strip().startswith(("PushedFilters", "ReadSchema"))]
print("📏 Optimisations Catalyst visibles :", pf)
codegen_ids = sorted(set(re.findall(r"codegen id : (\d+)", PLANS["dataframe_formatted"])), key=int)
print("📏 Blocs de code généré (whole-stage codegen) :", len(codegen_ids), "— ids", codegen_ids)

===== Lineage RDD (Partie 2)
(8) PythonRDD[1219] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[1218] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[1217] at partitionBy at <unknown>:0 []
 +-(8) PairwiseRDD[1216] at reduceByKey at /tmp/ipykernel_3165/2621212028.py:4 []
    |  PythonRDD[1215] at reduceByKey at /tmp/ipykernel_3165/2621212028.py:4 []
    |  MapPartitionsRDD[1214] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[1213] at partitionBy at <unknown>:0 []
    +-(8) PairwiseRDD[1212] at reduceByKey at /tmp/ipykernel_3165/2621212028.py:4 []
       |  PythonRDD[1211] at reduceByKey at /tmp/ipykernel_3165/2621212028.py:4 []
       |  s3a://lake/bronze/history/orders_history.jsonl MapPartitionsRDD[1210] at textFile at NativeMethodAccessorImpl.java:0 []
       |  s3a://lake/bronze/history/orders_history.jsonl HadoopRDD[1209] at textFile at NativeMethodAccessorImpl.java:0 []



===== Plan DataFrame — explain('formatted')
== Physical Plan ==
AdaptiveSparkPlan (26)
+- == Final Plan ==
   * HashAggregate (15)
   +- AQEShuffleRead (14)
      +- ShuffleQueryStage (13), Statistics(sizeInBytes=272.0 B, rowCount=8)
         +- Exchange (12)
            +- * HashAggregate (11)
               +- SortAggregate (10)
                  +- * Sort (9)
                     +- AQEShuffleRead (8)
                        +- ShuffleQueryStage (7), Statistics(sizeInBytes=3.3 MiB, rowCount=4.60E+4)
                           +- Exchange (6)
                              +- SortAggregate (5)
                                 +- * Sort (4)
                                    +- * Project (3)
                                       +- * Filter (2)
                                          +- Scan json  (1)
+- == Initial Plan ==
   HashAggregate (25)
   +- Exchange (24)
      +- HashAggregate (23)
         +- SortAggregate (22)
            +- Sort (21)
               +- Exchange (20)
  

## Étape 10 — Analyse technique (📘 Théorie)

### Catalyst Optimizer
Catalyst transforme une requête DataFrame/SQL en plan exécutable en quatre phases :
1. **Analyse** : le plan logique « non résolu » est résolu contre le catalogue et le schéma
   (noms de colonnes, types) → *Analyzed Logical Plan*.
2. **Optimisation logique** : application de règles indépendantes des données — *predicate pushdown*,
   *column pruning*, *constant folding*, simplification d'expressions, réordonnancement →
   *Optimized Logical Plan*.
3. **Planification physique** : génération de plans physiques candidats et choix par coût
   (ex. `BroadcastHashJoin` si une table est sous le seuil de broadcast, sinon `SortMergeJoin`).
4. **Génération de code** : Tungsten compile des étapes entières en bytecode JVM (*whole-stage codegen*).

Avec **AQE** (Adaptive Query Execution, actif par défaut depuis Spark 3.2), le plan est ré-optimisé
**pendant** l'exécution à partir des statistiques réelles des shuffles (fusion des petites partitions,
changement de stratégie de jointure, gestion du déséquilibre).

### Tungsten
Représentation binaire compacte des lignes (`UnsafeRow`, hors objets Java/Python), gestion mémoire explicite
et code généré : moins d'allocations, moins de GC, moins d'octets à sérialiser lors des shuffles.

### Pourquoi un schéma explicite
- pas de passe de lecture supplémentaire pour l'inférence (mesuré à l'étape 1) ;
- types exacts et stables (`int`, `timestamp`) au lieu de types devinés (`long`, `string`) ;
- détection des lignes non conformes (`_corrupt_record`) ;
- contrat clair entre producteurs et consommateurs (réutilisé en streaming à la Partie 4, où
  l'inférence est de toute façon impossible).

### RDD, DataFrame et Dataset
| | RDD | DataFrame | Dataset |
|---|---|---|---|
| Abstraction | collection d'objets distribuée | table de lignes typées par un schéma | collection d'objets typés + schéma |
| Typage | à l'exécution (Python) / compile (Scala) | à l'exécution (erreurs de colonne à l'analyse) | **à la compilation** (Scala/Java) |
| Optimisation | aucune : fonctions opaques | Catalyst + Tungsten | Catalyst + Tungsten (encodeurs) |
| Langages | Scala, Java, Python | Scala, Java, Python, R, SQL | **Scala et Java uniquement** |
| En PySpark | lambdas exécutées dans des processus Python | plan exécuté entièrement dans la JVM | non disponible |

### Limites de chaque approche
- **RDD** : pas d'optimiseur ; en PySpark, chaque élément traverse la frontière JVM ↔ Python
  (sérialisation pickle) ; mais contrôle total, utile pour des données non structurées ou des algorithmes
  ne s'exprimant pas en relationnel (et base de GraphX en Partie 6).
- **DataFrame** : expressivité limitée aux opérations relationnelles et fonctions intégrées ; une UDF Python
  réintroduit le coût de sérialisation et rend l'expression opaque pour Catalyst ; erreurs de nom de colonne
  détectées seulement à l'analyse (pas à la compilation en Python).
- **Dataset** : sécurité de typage, mais indisponible en Python et les lambdas typées (`map`) sont elles
  aussi opaques pour Catalyst.

### Biais de la comparaison (à dire en soutenance)
En PySpark, l'écart RDD/DataFrame mesure **deux effets mêlés** : l'optimisation Catalyst/Tungsten **et**
le coût du passage des données dans des processus Python (le parsing `json.loads` et les lambdas). En Scala,
où les RDD restent dans la JVM, l'écart serait plus faible. Le scénario B (données en cache) atténue mais
n'élimine pas ce biais.

## Étape 11 — Preuve de distribution et sauvegarde des résultats

In [16]:
tasks_by_host = {}
for e in rest("executors"):
    if e["id"] == "driver":
        continue
    ip = e["hostPort"].split(":")[0]
    try:
        host = socket.gethostbyaddr(ip)[0].split(".")[0]
    except OSError:
        host = ip
    tasks_by_host[host] = tasks_by_host.get(host, 0) + e["totalTasks"]
    print(f"executor {e['id']:<3} {e['hostPort']:<22} {host:<20} tâches={e['totalTasks']:>5} "
          f"CPU={e['totalDuration'] / 1000:>7.1f} s  shuffle écrit={e['totalShuffleWrite'] / 1e6:>6.2f} Mo")
print("\n📏 Tâches par worker :", tasks_by_host)
assert len(tasks_by_host) == 2 and all(v > 0 for v in tasks_by_host.values())

executor 3   172.22.0.8:33257       sbd-spark-worker-1   tâches=  302 CPU=   75.7 s  shuffle écrit= 14.56 Mo
executor 2   172.22.0.8:37055       sbd-spark-worker-1   tâches=  289 CPU=   75.2 s  shuffle écrit= 14.73 Mo
executor 1   172.22.0.9:42769       sbd-spark-worker-2   tâches=  307 CPU=   76.8 s  shuffle écrit= 17.26 Mo
executor 0   172.22.0.9:45157       sbd-spark-worker-2   tâches=  319 CPU=   76.9 s  shuffle écrit= 18.59 Mo

📏 Tâches par worker : {'sbd-spark-worker-1': 591, 'sbd-spark-worker-2': 626}


In [17]:
results = {
    "conditions": CONDITIONS,
    "verification_schema": {"lignes": n_lines, "non_conformes": n_corrupt, "nulls": nulls, "restaurants": n_rest,
                            "partitions_lecture": events.rdd.getNumPartitions()},
    "inference_schema": inference,
    "delivered": {"brut": n_delivered_raw, "uniques": n_delivered},
    "ca_panier_par_ville": [{"city": r.city, "ca_mad": round(r.ca_mad, 2), "commandes": r.commandes,
                             "panier_moyen_mad": round(r.panier_moyen_mad, 2)} for r in ca_city],
    "ca_total_mad": round(total, 2),
    "ca_categorie_ville": [r.asDict() for r in ca_cat_city],
    "top5_restaurants": [r.asDict() for r in top5],
    "controles": {"dataframe_egal_sql": same_sql, "egal_partie2_ca": same_p2_ca,
                  "egal_partie2_panier": same_p2_avg, "egal_partie2_top5": same_top},
    "distance_band": {"repartition": results_band["when/otherwise (natif)"], "temps_s": udf_times,
                      "medianes_s": {k: statistics.median(v) for k, v in udf_times.items()},
                      "operateurs_python": markers},
    "jointure_temps_s": join_times,
    "silver": {"chemin": SILVER, "lignes": n_silver, "ecriture_s": silver_write_s, "partitionnement": "city"},
    "taches_par_worker": tasks_by_host,
}
(LOGS / "10_part3_results.json").write_text(json.dumps(results, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
(LOGS / "10_part3_benchmark.json").write_text(json.dumps({"conditions": CONDITIONS, "protocole":
    "1 chauffe + 5 passes entrelacées avec rotation de l'ordre ; métriques ramenées à une exécution",
    **BENCH, "partie2_benchmark_rdd_s": json.loads((LOGS / "09_part2_metrics.json").read_text())["benchmark_ca_par_ville_rdd"]},
    indent=2, ensure_ascii=False), encoding="utf-8")
with open(LOGS / "10_part3_plans.txt", "w", encoding="utf-8") as fh:
    for k, v in PLANS.items():
        fh.write(f"\n{'=' * 100}\n== {k}\n{'=' * 100}\n{v}\n")
print("Écrits : 10_part3_results.json, 10_part3_benchmark.json, 10_part3_plans.txt")

Écrits : 10_part3_results.json, 10_part3_benchmark.json, 10_part3_plans.txt


In [18]:
spark.stop()
print("SparkSession arrêtée — DAG, stages et executors consultables dans le History Server : http://localhost:18080")

SparkSession arrêtée — DAG, stages et executors consultables dans le History Server : http://localhost:18080
